In [ ]:
from pathlib import Path
import os
import sys
import json

REPO_ROOT = (
    Path.cwd() if (Path.cwd() / "diffusion_project").is_dir() else Path.cwd().parent
)
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
# Set RESULTS_ROOT to the saved time-course results.
from pathlib import Path

RESULTS_ROOT = Path(os.environ.get("EXPERIMENT_ROOT", "./results")) / "time_course"
DATASETS = ["celeba64", "celeba32", "fashion_mnist", "cifar10", "mnist"]
MODEL_TYPES = ["dit", "unet"]
REPORT_EPOCHS = [1, 5, 10, 15, 20, 30, 50, 100, 200, 300, 400]
ZOOM_MAX_EPOCH = 50
CORR_ERROR_N = (
    400  # denominator for the 2 * sample std / sqrt(n) columns
)
REQUIRE_ALL_RUNS = (
    True  # False: plot means over available runs; tables always show counts
)
USE_SAVED_PNGS = False  # True: embed saved PNGs when present
# Default redraw uses saved numbers, so it also works after a job timed out before plotting.
SHOW_FULL_TABLES = True
SHOW_PER_RUN_TABLES = True
DISPLAY_WIDTH = 850
FIGSIZE = (10.4, 7.6)
DPI = 150
CORR_YLIM = (0.75, 0.95)
MSE_YLIM = (0.02, 0.16)
ZOOM_CORR_YLIM = (0.75, 0.95)
ZOOM_MSE_YLIM = (0.02, 0.16)
LEGEND_FONT_SIZE = 18
LEGEND_OUTSIDE = False  # True puts every legend to the right, outside the trajectories
# Each plot can have its own location: 'best', 'upper left', 'lower right', etc.
LEGEND_LOCATIONS = {
    "dit_corr_full": "lower right",
    "dit_corr_first50": "lower right",
    "dit_mse_full": "upper right",
    "dit_mse_first50": "upper right",
    "unet_corr_full": "lower right",
    "unet_corr_first50": "lower right",
    "unet_mse_full": "upper right",
    "unet_mse_first50": "upper right",
}

# Per-image numbers for one particular evaluation (last section).
DETAIL_DATASET = "celeba32"
DETAIL_MODEL = "dit"
DETAIL_SUBSET = 1
DETAIL_SUFFIX = ""  # '', '_b', or '_c'
DETAIL_EPOCH = 15

DATASET_LABELS = {
    "celeba64": "CelebA64",
    "celeba32": "CelebA32",
    "fashion_mnist": "FashionMNIST",
    "cifar10": "CIFAR10",
    "mnist": "MNIST",
}
DATASET_COLORS = {
    "celeba64": "C0",
    "fashion_mnist": "C1",
    "cifar10": "C2",
    "mnist": "C3",
    "celeba32": "C4",
}
MODEL_LABELS = {"dit": "DiT", "unet": "UNet"}
# Reference configuration for the optional checkpoint audit.
EXPECTED_COMMON = {"epochs": 400, "lr": 2e-4, "weight_decay": 5e-2}
EXPECTED_BY_MODEL = {
    "dit": {
        "patch_size": 4,
        "hidden_size": 512,
        "depth": 12,
        "num_heads": 8,
        "mlp_ratio": 4.0,
        "batch_size": 32,
    },
    "unet": {
        "model_channels": 128,
        "channel_mult": [1, 2, 2, 4],
        "num_heads": 8,
        "attention_resolutions": [16],
        "batch_size": 32,
    },
}


The error columns use pooled per-image sample standard deviations (`ddof=1`) and the configurable denominator `CORR_ERROR_N=400`. Actual finite image counts are reported separately. These are descriptive error estimates, not confidence intervals for the median or adjustments for repeated seeds.


# Time-course Figure 2: results viewer

**Run All, then save this notebook** to retain the embedded figures and tables.
No GPU, Torch, model checkpoints, training data, or inference is needed. This
notebook reads the `trajectories.npz` files written by the batch job. It does not
modify those files or submit jobs. Rerun it to refresh results while jobs finish.

The default recreates the eight final plots from saved metrics: full and first-50
epoch corr./MSE plots for both DiT and UNet. Set `USE_SAVED_PNGS=True` to embed the
saved figures instead (these may reflect an earlier state of the batch;
the viewer's filtering settings do not change those images).

Correlation is signed Pearson correlation per image. Each model's median over
images is computed first; the curve averages those medians over runs. MSE is the
mean across pixels/images/runs in the normalized image units. Missing
results and undefined constant-image correlations remain NaN. Tables include
available-run counts; `REQUIRE_ALL_RUNS` controls plots and best/last summaries.
Standard deviations are descriptive spreads, not confidence intervals.

In [ ]:
import io
import json
import html
import warnings
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image

if CORR_ERROR_N <= 0:
    raise ValueError("CORR_ERROR_N must be positive")
RESULTS_ROOT = RESULTS_ROOT.expanduser()
if not RESULTS_ROOT.is_dir():
    raise FileNotFoundError(
        f"Results directory not found: {RESULTS_ROOT}. Set RESULTS_ROOT in the first cell."
    )
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 26,
        "axes.labelsize": 30,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def finite_values(values):
    values = np.asarray(values)
    return values[np.isfinite(values)]


def mean(values):
    valid = finite_values(values)
    return float(valid.mean()) if len(valid) else float("nan")


def median(values):
    valid = finite_values(values)
    return float(np.median(valid)) if len(valid) else float("nan")


def std(values):
    valid = finite_values(values)
    return float(valid.std(ddof=1)) if len(valid) > 1 else float("nan")


def show_table(rows, title, collapsible=False):
    if not rows:
        print(title + ": no rows available.")
        return

    def format_value(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.7g}" if np.isfinite(value) else "—"
        return str(value)

    columns = list(rows[0])
    header = "".join(
        '<th style="text-align:left;padding:5px">' + html.escape(c) + "</th>"
        for c in columns
    )
    body = "".join(
        "<tr>"
        + "".join(
            '<td style="padding:4px 8px;white-space:nowrap">'
            + html.escape(format_value(row.get(c, "")))
            + "</td>"
            for c in columns
        )
        + "</tr>"
        for row in rows
    )
    table = (
        '<div style="max-height:550px;overflow:auto"><table><thead><tr>'
        + header
        + "</tr></thead><tbody>"
        + body
        + "</tbody></table></div>"
    )
    title = html.escape(title)
    display(
        HTML(
            (
                "<details><summary>"
                + title
                + f" ({len(rows)} rows)</summary>"
                + table
                + "</details>"
            )
            if collapsible
            else "<h4>" + title + "</h4>" + table
        )
    )


results = (
    {}
)  # (dataset, model) -> raw per-image arrays, epoch/run labels, metadata, checkpoint provenance
summary_rows = []
per_run_rows = []
status_rows = []
run_status_rows = []
for kind in MODEL_TYPES:
    for dataset in DATASETS:
        path = RESULTS_ROOT / dataset / kind / "trajectories.npz"
        if not path.exists():
            status_rows.append(
                {
                    "dataset": dataset,
                    "model": kind,
                    "saved evaluations": 0,
                    "expected evaluations": "unknown",
                    "complete epochs": 0,
                    "last complete epoch": "—",
                    "status": "missing trajectories.npz",
                }
            )
            continue
        with np.load(path, allow_pickle=False) as saved:
            metadata = json.loads(str(saved["metadata_json"].item()))
            data = {
                "epochs": saved["epochs"].astype(int),
                "corr": saved["per_image_corr"].copy(),
                "mse": saved["per_image_mse"].copy(),
                "completed": saved["completed"].astype(bool),
                "metadata": metadata,
                "checkpoints": json.loads(str(saved["checkpoints_json"].item())),
                "runs": [tuple(run) for run in metadata["settings"]["runs"]],
            }
        epochs, runs, done = data["epochs"], data["runs"], data["completed"]
        if (
            data["corr"].shape != data["mse"].shape
            or data["corr"].ndim != 3
            or done.shape != (len(runs), len(epochs))
            or data["corr"].shape[:2] != done.shape
        ):
            raise ValueError(f"{path}: inconsistent run/epoch/image shapes")
        data["corr"][~done] = np.nan
        data["mse"][~done] = np.nan
        run_corr = np.full(done.shape, np.nan)
        run_mse = np.full(done.shape, np.nan)
        for i, (subset, suffix) in enumerate(runs):
            evaluated = epochs[done[i]]
            run_status_rows.append(
                {
                    "dataset": dataset,
                    "model": kind,
                    "subset": subset,
                    "suffix": suffix or "(base)",
                    "saved epochs": len(evaluated),
                    "requested epochs": len(epochs),
                    "last saved epoch": int(evaluated[-1]) if len(evaluated) else "—",
                }
            )
            for j, epoch in enumerate(epochs):
                if done[i, j]:
                    run_corr[i, j] = median(data["corr"][i, j])
                    run_mse[i, j] = mean(data["mse"][i, j])
                per_run_rows.append(
                    {
                        "dataset": dataset,
                        "model": kind,
                        "subset": subset,
                        "run_suffix": suffix,
                        "epoch": int(epoch),
                        "completed": bool(done[i, j]),
                        "median_corr": run_corr[i, j],
                        "mean_mse": run_mse[i, j],
                        "valid_corr_images": int(np.isfinite(data["corr"][i, j]).sum()),
                        "corr_std_images": std(data["corr"][i, j]),
                        "mse_std_images": std(data["mse"][i, j]),
                    }
                )
        rows = []
        for j, epoch in enumerate(epochs):
            rows.append(
                {
                    "dataset": dataset,
                    "model": kind,
                    "epoch": int(epoch),
                    "mean_median_corr": mean(run_corr[:, j]),
                    "corr_2sigma_sqrt_n": 2
                    * std(data["corr"][:, j, :])
                    / np.sqrt(CORR_ERROR_N),
                    "mean_mse": mean(run_mse[:, j]),
                    "mse_2sigma_sqrt_n": 2
                    * std(data["mse"][:, j, :])
                    / np.sqrt(CORR_ERROR_N),
                    "error_n": CORR_ERROR_N,
                    "n_corr_images": int(np.isfinite(data["corr"][:, j, :]).sum()),
                    "n_mse_images": int(np.isfinite(data["mse"][:, j, :]).sum()),
                    "std_run_median_corr": std(run_corr[:, j]),
                    "std_run_mean_mse": std(run_mse[:, j]),
                    "n_corr_runs": int(np.isfinite(run_corr[:, j]).sum()),
                    "n_mse_runs": int(np.isfinite(run_mse[:, j]).sum()),
                    "n_completed_runs": int(done[:, j].sum()),
                    "n_requested_runs": len(runs),
                }
            )
        data["summary"] = rows
        results[dataset, kind] = data
        summary_rows.extend(rows)
        complete = epochs[done.all(axis=0)]
        status_rows.append(
            {
                "dataset": dataset,
                "model": kind,
                "saved evaluations": int(done.sum()),
                "expected evaluations": int(done.size),
                "complete epochs": len(complete),
                "last complete epoch": int(complete[-1]) if len(complete) else "—",
                "status": "complete" if done.all() else "partial",
            }
        )

show_table(status_rows, "Saved progress (read directly from trajectories)")
show_table(run_status_rows, "Progress for every trained model", collapsible=True)
if not results:
    print(
        "No trajectory files found. Check RESULTS_ROOT and the batch logs. Saved PNGs can still be displayed below."
    )
if any(r["status"] != "complete" for r in status_rows):
    print(
        "Some results are missing or partial. With REQUIRE_ALL_RUNS=True, plots show only epochs with valid metrics for every selected run."
    )


## All final visualizations
These images are embedded into notebook outputs. Save the notebook after running
to keep them visible when reopening or sharing it.
The first cell sets corr. limits to 0.75–0.95 and MSE limits to 0.02–0.16,
including the zooms. `LEGEND_LOCATIONS` lets you move each plot's legend
independently; `LEGEND_OUTSIDE=True` moves all legends outside the axes.
Keep `USE_SAVED_PNGS=False` for these controls to apply. Values outside the
chosen limits are clipped visually; the numerical tables retain them.


In [ ]:
def eligible(row, metric):
    count = "n_corr_runs" if metric == "mean_median_corr" else "n_mse_runs"
    return np.isfinite(row[metric]) and (
        not REQUIRE_ALL_RUNS or row[count] == row["n_requested_runs"]
    )


for kind in MODEL_TYPES:
    for metric, short_name, ylabel, limits, zoom_limits in (
        ("mean_median_corr", "corr", "Mean median corr.", CORR_YLIM, ZOOM_CORR_YLIM),
        ("mean_mse", "mse", "MSE", MSE_YLIM, ZOOM_MSE_YLIM),
    ):
        for zoom in (False, True):
            name = f'{kind}_{short_name}_{"first" + str(ZOOM_MAX_EPOCH) if zoom else "full"}'
            png = RESULTS_ROOT / "plots" / (name + ".png")
            print(name)
            if (USE_SAVED_PNGS or not results) and png.exists():
                display(Image(data=png.read_bytes(), format="png", width=DISPLAY_WIDTH))
                continue
            fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
            plotted = False
            for dataset in DATASETS:
                data = results.get((dataset, kind))
                if data is None:
                    continue
                rows = [
                    r
                    for r in data["summary"]
                    if not zoom or r["epoch"] <= ZOOM_MAX_EPOCH
                ]
                values = [r[metric] if eligible(r, metric) else np.nan for r in rows]
                if not np.isfinite(values).any():
                    continue
                ax.plot(
                    [r["epoch"] for r in rows],
                    values,
                    linewidth=4.4,
                    color=DATASET_COLORS.get(dataset),
                    label=DATASET_LABELS.get(dataset, dataset),
                )
                plotted = True
            ax.set_xlabel("Epoch")
            ax.set_ylabel(ylabel)
            ax.set_title(
                MODEL_LABELS.get(kind, kind)
                + (f" (first {ZOOM_MAX_EPOCH} epochs)" if zoom else "")
            )
            ax.grid(alpha=0.25, linewidth=0.8)
            if zoom:
                ax.set_xlim(0, ZOOM_MAX_EPOCH)
            ylim = zoom_limits if zoom else limits
            if ylim is not None:
                ax.set_ylim(*ylim)
            if plotted:
                if LEGEND_OUTSIDE:
                    ax.legend(
                        fontsize=LEGEND_FONT_SIZE,
                        loc="upper left",
                        bbox_to_anchor=(1.02, 1),
                    )
                else:
                    ax.legend(
                        fontsize=LEGEND_FONT_SIZE,
                        loc=LEGEND_LOCATIONS.get(name, "best"),
                    )
            else:
                ax.text(
                    0.5,
                    0.5,
                    "No eligible saved results",
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    fontsize=18,
                )
            fig.tight_layout()
            buffer = io.BytesIO()
            fig.savefig(buffer, format="png", dpi=DPI)
            plt.close(fig)
            display(Image(data=buffer.getvalue(), format="png", width=DISPLAY_WIDTH))


## Numerical summaries
The selected-epoch tables report available-run means even when some runs are
missing; always check the count columns. Best/last summaries use the same
`REQUIRE_ALL_RUNS` rule as the plots and report actual evaluated epochs, without
interpolation. All stored precision remains accessible in the Python variables;
tables round only the displayed values.

`corr_2sigma_sqrt_n` is `2 * sigma / sqrt(CORR_ERROR_N)`, with
`CORR_ERROR_N=400` by default (so the column equals `sigma / 10`). Here `sigma`
is the **sample** standard deviation (`ddof=1`) of the individual image
correlations pooled over available completed runs at that epoch. Non-finite values are excluded.
The denominator stays 400 even when a different number of correlations is
available. This is a descriptive error estimate, not a confidence interval for the mean of per-run medians.


In [ ]:
selected_epoch_rows = [r for r in summary_rows if r["epoch"] in REPORT_EPOCHS]
for kind in MODEL_TYPES:
    show_table(
        [r for r in selected_epoch_rows if r["model"] == kind],
        MODEL_LABELS.get(kind, kind) + ": metrics at requested epochs",
    )

best_rows = []
for (dataset, kind), data in results.items():
    for metric, description, count_key, maximize in (
        ("mean_median_corr", "corr.", "n_corr_runs", True),
        ("mean_mse", "MSE", "n_mse_runs", False),
    ):
        for scope, cutoff in (
            ("full", None),
            (f"first {ZOOM_MAX_EPOCH}", ZOOM_MAX_EPOCH),
        ):
            valid = [
                r
                for r in data["summary"]
                if eligible(r, metric) and (cutoff is None or r["epoch"] <= cutoff)
            ]
            if not valid:
                continue
            best = (max if maximize else min)(valid, key=lambda r: r[metric])
            last = max(valid, key=lambda r: r["epoch"])
            best_rows.append(
                {
                    "dataset": dataset,
                    "model": kind,
                    "range": scope,
                    "metric": description,
                    "best_epoch": best["epoch"],
                    "best_value": best[metric],
                    "best_n_runs": best[count_key],
                    "last_epoch": last["epoch"],
                    "last_value": last[metric],
                    "last_n_runs": last[count_key],
                    "requested_runs": last["n_requested_runs"],
                }
            )
show_table(best_rows, "Best and last available values (ties choose the earliest epoch)")

if SHOW_FULL_TABLES:
    for (dataset, kind), data in results.items():
        show_table(data["summary"], f"{dataset} / {kind}: all epochs", collapsible=True)
if SHOW_PER_RUN_TABLES:
    for dataset, kind in results:
        show_table(
            [r for r in per_run_rows if r["dataset"] == dataset and r["model"] == kind],
            f"{dataset} / {kind}: all individual run metrics",
            collapsible=True,
        )


## Per-image values and provenance
Change `DETAIL_*` in the first cell for a different evaluation. Full raw arrays
are available as `results[dataset, model]['corr']` and `['mse']`, indexed by
[run, epoch, image]; the accompanying `runs` and `epochs` give their labels.
`summary_rows`, `per_run_rows`, `selected_epoch_rows`, and `best_rows` contain all
the table data. Saved checkpoint configurations and averaging epochs are in
`results[dataset, model]['checkpoints']`.

In [ ]:
detail = results.get((DETAIL_DATASET, DETAIL_MODEL))
detail_rows = []
if detail is None:
    print("No saved data for the selected DETAIL_DATASET / DETAIL_MODEL.")
elif (DETAIL_SUBSET, DETAIL_SUFFIX) not in detail["runs"] or DETAIL_EPOCH not in detail[
    "epochs"
]:
    print("Requested detail run or epoch is not in the saved analysis grid.")
else:
    i = detail["runs"].index((DETAIL_SUBSET, DETAIL_SUFFIX))
    j = int(np.flatnonzero(detail["epochs"] == DETAIL_EPOCH)[0])
    if not detail["completed"][i, j]:
        print("This particular run/epoch has not been evaluated yet.")
    else:
        detail_rows = [
            {"image_index": k, "corr.": float(c), "MSE": float(m)}
            for k, (c, m) in enumerate(zip(detail["corr"][i, j], detail["mse"][i, j]))
        ]
        show_table(
            detail_rows,
            f"{DETAIL_DATASET} / {DETAIL_MODEL} / s{DETAIL_SUBSET}{DETAIL_SUFFIX} / epoch {DETAIL_EPOCH}",
        )
        checkpoint = detail["checkpoints"].get(f"{i}:{DETAIL_EPOCH}", {})
        print(
            "Averaged checkpoints:", checkpoint.get("averaged_epochs", "not recorded")
        )
        print("Noise:", detail["metadata"].get("seed_file"))
        print(
            "Reference:",
            detail["metadata"].get("references", {}).get(str(DETAIL_SUBSET), {}),
        )
        print("Analysis settings:", detail["metadata"]["settings"])


# Checkpoint configuration audit

Read the saved training configuration for each run. Expected values are editable in the first cell. Batches per epoch use floor(subset_size / batch_size), matching drop_last=True.


In [ ]:
audit_rows = []
for (dataset, kind), data in results.items():
    for run_index, (subset, suffix) in enumerate(data["runs"]):
        groups = {}
        for key, record in data["checkpoints"].items():
            saved_run, epoch = (int(x) for x in key.split(":"))
            config = record.get("config")
            if saved_run != run_index or not config:
                continue
            signature = json.dumps(config, sort_keys=True)
            groups.setdefault(signature, {"config": config, "epochs": []})[
                "epochs"
            ].append(epoch)
        for group in list(groups.values()) or [{"config": {}, "epochs": []}]:
            config = group["config"]
            epoch_ids = sorted(group["epochs"])
            expected = dict(EXPECTED_COMMON, **EXPECTED_BY_MODEL[kind])
            differences = []
            for key, value in expected.items():
                if key not in config:
                    differences.append(f"{key}: unknown")
                elif config[key] != value:
                    differences.append(f"{key}: {config[key]} (expected {value})")
            subset_size = data["metadata"]["settings"].get("subset_size")
            if subset_size is None:
                differences.append("subset_size: unknown")
            elif subset_size != 10000:
                differences.append(f"subset_size: {subset_size} (expected 10000)")
            batch_size = config.get("batch_size")
            steps = subset_size // batch_size if subset_size and batch_size else None
            expected_steps = (
                subset_size // expected["batch_size"] if subset_size else None
            )
            widths = (
                [config["model_channels"] * m for m in config["channel_mult"]]
                if kind == "unet"
                and "model_channels" in config
                and "channel_mult" in config
                else None
            )
            audit_rows.append(
                {
                    "dataset": dataset,
                    "model": kind,
                    "subset": subset,
                    "run_suffix": suffix,
                    "evaluated_epochs": (
                        f"{epoch_ids[0]}–{epoch_ids[-1]} ({len(epoch_ids)} saved)"
                        if epoch_ids
                        else "none with config"
                    ),
                    "image_size": config.get("image_size", "unknown"),
                    "DiT_blocks": config.get("depth", "—"),
                    "DiT_hidden": config.get("hidden_size", "—"),
                    "UNet_widths": widths if widths is not None else "—",
                    "heads": config.get("num_heads", "unknown"),
                    "batch_size": batch_size or "unknown",
                    "batches_per_epoch": steps if steps is not None else "unknown",
                    "batch_count_vs_expected": (
                        steps / expected_steps
                        if expected_steps and steps is not None
                        else float("nan")
                    ),
                    "differences_from_expected": (
                        "; ".join(differences)
                        if differences
                        else "matches compared config fields"
                    ),
                }
            )
show_table(audit_rows, "Checkpoint configuration audit")


In [ ]:
# Export exactly the displayed numbers, preserving full precision in CSV.
import csv

for name, rows in [("summary", summary_rows), ("per_run_metrics", per_run_rows)]:
    if rows:
        with (RESULTS_ROOT / f"{name}_viewer.csv").open("w") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
